In [1]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from pymc.math import dot, invlogit

rng = np.random.default_rng(6420)

In [2]:
data_frame = pd.read_csv('data_clean/bfy_age4_clean.csv')

In [3]:
data_frame.shape

(1000, 35)

In [4]:
continuous_vars = ['birthweight_oz', 'gest_age', 'mom_age', 'mom_health', 'cesd', 'n_children', 'n_adults', 'hh_income', 'net_worth']

In [5]:
binary = ['female', 'smoked', 'alcohol', 'biodad_in_hh', 'spanish', 'under4']

In [6]:
# Standardize helper:

def standardize(dataframe):
    continuous = [col for col in dataframe if dataframe[col].nunique() > 2]
    standardized = dataframe.copy()

    standardized[continuous] = (dataframe[continuous] - dataframe[continuous].mean()) / (2 * dataframe[continuous].std())

    return standardized

In [7]:
covariates = ['birthweight_oz', 'gest_age', 'mom_age', 'mom_health', 'cesd', 'n_children', 'n_adults', 'hh_income', 'net_worth', 'female', 'smoked', 'alcohol', 'biodad_in_hh', 'spanish', 'under4']

In [8]:
df_standard = standardize(data_frame[covariates])
covs = df_standard.to_numpy(copy=True)

missing = np.isnan(covs)
covs[np.isnan(covs)] = 0

In [9]:
covs.shape, missing.shape, missing.sum(axis=0)

((1000, 15),
 (1000, 15),
 array([ 24,   5,   0,   0,  10,   0,   0,   0, 111,   0,   0,   0,   0,
        168, 108]))

In [10]:
categorical_vars = ['site', 'mom_educ', 'mom_race', 'marital', 'assessor']

parts = [covs, missing]
for elem in categorical_vars: 
    x = data_frame[elem].to_numpy(copy=True)
    x[np.isnan(x)] = 99
    parts.append(pd.get_dummies(x).to_numpy()[:, 1:])

matrix = np.concatenate(parts, axis=1)

matrix.shape

(1000, 67)

In [11]:
treat = data_frame['treat'].to_numpy(copy=True)
y = data_frame['vocab_z'].to_numpy(copy=True)

missing_mask = np.isnan(y)
y_vocab = y[~missing_mask]
treat_vocab = treat[~missing_mask]
matrix_vocab = matrix[~missing_mask]

varies_mask = matrix_vocab.std(axis=0) > 0
matrix_vocab = matrix_vocab[:, varies_mask]

y_vocab.shape, treat_vocab.shape, matrix_vocab.shape

((700,), (700,), (700, 55))

In [12]:
with pm.Model() as model_vocab:
    tau = pm.Gamma('tau', 0.001, 0.001)
    alpha = pm.Normal('intercept', 0, tau=0.001)
    delta = pm.Normal('delta', 0, tau=0.001)
    beta = pm.Normal('beta', 0, tau = 0.001, shape=matrix_vocab.shape[1])

    variance = pm.Deterministic('variance', 1 / tau)

    mu = alpha + delta * treat_vocab + pm.math.dot(matrix_vocab, beta)

    pm.Normal('likelihood', mu=mu, tau= tau, observed=y_vocab)

    trace = pm.sample(3000, random_seed=6420)


NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [13]:
az.summary(trace, var_names='~beta', ci_prob=0.95)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
intercept,-0.04,0.37,-0.73,0.72,793,1282,1.01,0.013,0.0066
delta,-0.057,0.071,-0.2,0.08,13575,9607,1.00,0.00061,0.00061
tau,1.233,0.068,1.1,1.4,14838,9532,1.00,0.00056,0.00062
variance,0.8136,0.0449,0.73,0.91,14838,9532,1.00,0.00037,0.00041


In [14]:
outcomes = ['vocab_z', 'ef_z', 'cbcl_z', 'eeg_z', 'matrices_z', 'reading_z']

summaries = []

for out in outcomes: 
    y = data_frame[out].to_numpy(copy=True)
    missing_mask = np.isnan(y)
    y_obs = y[~missing_mask]
    treat_obs = treat[~missing_mask]
    matrix_obs = matrix[~missing_mask]

    varies_mask = matrix_obs.std(axis=0) > 0
    matrix_obs = matrix_obs[:, varies_mask]

    with pm.Model() as model:
        tau = pm.Gamma('tau', 0.001, 0.001)
        alpha = pm.Normal('intercept', 0, tau=0.001)
        delta = pm.Normal('delta', 0, tau=0.001)
        beta = pm.Normal('beta', 0, tau=0.001, shape=matrix_obs.shape[1])
        variance = pm.Deterministic('variance', 1 / tau)

        prob_positive = pm.Deterministic('prob_positive', pm.math.switch(delta >= 0, 1, 0))

        mu = alpha + delta * treat_obs + pm.math.dot(matrix_obs, beta)
        pm.Normal('likelihood', mu=mu, tau=tau, observed=y_obs)

        trace = pm.sample(3000, random_seed=6420)

    summaries.append(az.summary(trace, var_names=['delta', 'prob_positive'], ci_prob=0.95))


NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

... (2 outputs left out)

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

NUTS[nutpie]: [tau, intercept, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [15]:
for out, summary in zip(outcomes, summaries):
    print(out)
    print(summary)
    print("\n")

vocab_z
                 mean     sd eti95_lb eti95_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
delta          -0.057  0.071     -0.2     0.08    13575     9607  1.00   0.00061  0.00061
prob_positive   0.213  0.409        0        1    10082    10082  1.00    0.0041   0.0029


ef_z
                mean     sd eti95_lb eti95_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
delta          0.056  0.074   -0.088      0.2    13050     8873  1.00   0.00065  0.00067
prob_positive  0.771   0.42        0        1     9533     9533  1.00    0.0043   0.0028


cbcl_z
                mean     sd eti95_lb eti95_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
delta          0.038  0.066   -0.089     0.17    14268     9197  1.00   0.00055  0.00063
prob_positive  0.713  0.452        0        1    11808    11808  1.00    0.0042    0.002


eeg_z
                mean     sd eti95_lb eti95_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
delta          0.101  0.078    -0.05     0.26    13254     9155  1.00   0.0

In [16]:
# Diagnosis model

y = data_frame['diagnosis'].to_numpy(copy=True)
missing_mask = np.isnan(y)
y_obs = y[~missing_mask]
treat_obs = treat[~missing_mask]
matrix_obs = matrix[~missing_mask]

matrix_obs = matrix_obs[:, :-17]

varies_mask = matrix_obs.std(axis=0) > 0
matrix_obs = matrix_obs[:, varies_mask]

y_obs.shape, treat_obs.shape, matrix_obs.shape, y_obs.sum()

((881,), (881,), (881, 40), np.float64(124.0))

In [17]:
with pm.Model() as diag_model:
    alpha = pm.Normal('alpha', mu=0, sigma=10)
    delta = pm.Normal('delta', mu=0, sigma=5)
    betas = pm.Normal('beta', mu=0, sigma=5, shape=matrix_obs.shape[1])

    prob = invlogit(alpha + delta * treat_obs + dot(matrix_obs, betas))

    cohen_d = pm.Deterministic('cohen_d', delta * np.sqrt(3) / np.pi)

    prob_positive = pm.Deterministic('prob_positive', pm.math.switch(delta >= 0, 1, 0))

    pm.Bernoulli('y', p=prob, observed = y_obs)

    trace_diagnosis = pm.sample(5000, random_seed = 6420)

NUTS[nutpie]: [alpha, delta, beta]


/Users/abtinkhashayar/pymc_env/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [18]:
az.summary(trace_diagnosis, var_names=['delta', 'cohen_d', 'prob_positive'], ci_prob=0.95)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
delta,0.125,0.22,-0.31,0.56,33337,15449,1.00,0.0012,0.0017
cohen_d,0.069,0.121,-0.17,0.31,33337,15449,1.00,0.00066,0.00093
prob_positive,0.716,0.451,0,1,20649,20000,1.00,0.0031,0.0015
